# Run single example

Notebook version of `llm_run.py`. Reuses its functions directly (same prompt, same
message-building, same JSON answer parsing) so there's a single source of truth for the
prompt/logic -- this notebook only adds the ability to run one scenario at a time and to
print each model response in full.

Run the setup cells once, then use either the **single scenario** cell or the **all
scenarios** cell as many times as you like.

In [1]:
import os
import sys

# Jupyter starts the kernel with cwd = this notebook's own directory (evaluation/), and
# some Jupyter setups back sys.path[0] with a *dynamic* '' entry that tracks the current
# cwd rather than a fixed path -- so once we chdir below, "import llm_run" could stop
# finding evaluation/llm_run.py. Pin this directory onto sys.path explicitly first.
_notebook_dir = os.getcwd()
if _notebook_dir not in sys.path:
    sys.path.insert(0, _notebook_dir)

# .env and dataset/ live at the repo root, one level up. Walk up until we find .env (the
# repo root marker) so the relative paths llm_run.py uses (ENV_FILE=".env",
# DATASET_DIR="dataset") resolve correctly. Idempotent: safe to re-run this cell.
_MAX_LEVELS_UP = 5
for _ in range(_MAX_LEVELS_UP):
    if os.path.isfile(".env"):
        break
    os.chdir("..")
else:
    raise RuntimeError("Could not find .env by walking up from the notebook's directory")
print("Working directory:", os.getcwd())

Working directory: /home/tejad/CodingProjects/Aviation-Emergencies-Benchmark


In [2]:
import json
from collections import defaultdict

import llm_run

## Setup

Loads the API key, discovers every `scenario.json` under `dataset/`, and lists them with
their index so you can pick one below.

In [3]:
MODEL = llm_run.DEFAULT_MODEL
RUNS =  1 # llm_run.RUNS_PER_SCENARIO

api_key = llm_run.load_api_key()
scenario_paths = llm_run.find_scenarios(llm_run.DATASET_DIR)

for i, path in enumerate(scenario_paths):
    print(f"[{i}] {path}")

[0] dataset/example_0/scenario.json
[1] dataset/example_1/scenario.json
[2] dataset/example_10/scenario.json
[3] dataset/example_11/scenario.json
[4] dataset/example_12/scenario.json
[5] dataset/example_13/scenario.json
[6] dataset/example_14/scenario.json
[7] dataset/example_15/scenario.json
[8] dataset/example_16/scenario.json
[9] dataset/example_17/scenario.json
[10] dataset/example_18/scenario.json
[11] dataset/example_19/scenario.json
[12] dataset/example_2/scenario.json
[13] dataset/example_20/scenario.json
[14] dataset/example_21/scenario.json
[15] dataset/example_22/scenario.json
[16] dataset/example_23/scenario.json
[17] dataset/example_24/scenario.json
[18] dataset/example_25/scenario.json
[19] dataset/example_26/scenario.json
[20] dataset/example_27/scenario.json
[21] dataset/example_28/scenario.json
[22] dataset/example_29/scenario.json
[23] dataset/example_3/scenario.json
[24] dataset/example_30/scenario.json
[25] dataset/example_31/scenario.json
[26] dataset/example_32/sc

In [4]:
def rename_scenarios_by_creation_order(scenario_paths):
    """Renames each scenario's containing folder in place to scenario_<i> (i=0, 1, 2, ...),
    ordered by the scenario's created_at field, so folder names become simple sequential
    labels instead of timestamps. Returns the new scenario.json paths in that same order.

    Two-phase (rename to a temp name, then to the final name) so this is safe to re-run:
    a final scenario_i name might already be occupied by a not-yet-renamed folder.
    """
    entries = []
    for path in scenario_paths:
        scenario = json.loads(path.read_text())
        entries.append((scenario.get("created_at", ""), path))
    entries.sort(key=lambda entry: entry[0])

    temp_dirs = []
    for i, (_, old_scenario_path) in enumerate(entries):
        old_dir = old_scenario_path.parent
        temp_dir = old_dir.parent / f"__renaming_{i}"
        old_dir.rename(temp_dir)
        temp_dirs.append(temp_dir)

    new_paths = []
    for i, temp_dir in enumerate(temp_dirs):
        final_dir = temp_dir.parent / f"scenario_{i}"
        temp_dir.rename(final_dir)
        new_paths.append(final_dir / "scenario.json")
    return new_paths


scenario_paths = rename_scenarios_by_creation_order(scenario_paths)
for i, path in enumerate(scenario_paths):
    print(f"[{i}] {path}")

[0] dataset/scenario_0/scenario.json
[1] dataset/scenario_1/scenario.json
[2] dataset/scenario_2/scenario.json
[3] dataset/scenario_3/scenario.json
[4] dataset/scenario_4/scenario.json
[5] dataset/scenario_5/scenario.json
[6] dataset/scenario_6/scenario.json
[7] dataset/scenario_7/scenario.json
[8] dataset/scenario_8/scenario.json
[9] dataset/scenario_9/scenario.json
[10] dataset/scenario_10/scenario.json
[11] dataset/scenario_11/scenario.json
[12] dataset/scenario_12/scenario.json
[13] dataset/scenario_13/scenario.json
[14] dataset/scenario_14/scenario.json
[15] dataset/scenario_15/scenario.json
[16] dataset/scenario_16/scenario.json
[17] dataset/scenario_17/scenario.json
[18] dataset/scenario_18/scenario.json
[19] dataset/scenario_19/scenario.json
[20] dataset/scenario_20/scenario.json
[21] dataset/scenario_21/scenario.json
[22] dataset/scenario_22/scenario.json
[23] dataset/scenario_23/scenario.json
[24] dataset/scenario_24/scenario.json
[25] dataset/scenario_25/scenario.json
[26] d

In [5]:
def run_scenario(scenario_path, model, runs):
    """Runs `model` on one scenario `runs` times, printing the full raw response each time.
    Returns (outcomes, tags) where outcomes is a list of "correct"/"wrong"/"unparseable"."""
    scenario = json.loads(scenario_path.read_text())
    image_path = scenario_path.parent / scenario["image_file"]
    correct_number = llm_run.correct_option_number(scenario)
    tags = scenario.get("starting_condition_tags", []) + scenario.get("expected_behavior_tags", [])

    print(f"=== {scenario_path} ===")
    if correct_number is None:
        print("SKIPPED: no ground_truth_index set")
        return [], tags

    messages = llm_run.build_messages(scenario, image_path)
    outcomes = []
    for run_index in range(1, runs + 1):
        label = f"run {run_index}/{runs}"
        response_text = llm_run.call_openrouter(model, messages, api_key)
        print(f"--- {label}: model response ---")
        print(response_text)

        chosen_number = llm_run.parse_answer(response_text)
        if chosen_number is None:
            print(f"{label}: UNPARSEABLE (no integer \"answer\" field found in JSON response)")
            outcomes.append("unparseable")
        elif chosen_number == correct_number:
            print(f"{label}: chose #{chosen_number}, correct #{correct_number} -> CORRECT")
            outcomes.append("correct")
        else:
            print(f"{label}: chose #{chosen_number}, correct #{correct_number} -> WRONG")
            outcomes.append("wrong")
        print()
    return outcomes, tags

## Run a single scenario

Set `SCENARIO_INDEX` to one of the indices printed in the setup cell above, then run this
cell. Prints the model's full response for every run, plus a per-run correctness verdict.

In [ ]:
SCENARIO_INDEX = 7  # change this to pick a different scenario from the list above

outcomes, _tags = run_scenario(scenario_paths[SCENARIO_INDEX], MODEL, RUNS)
print(f"Summary: {'/'.join(o.upper() for o in outcomes)}")